<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">
</p>

# **NOTEBOOK 8.2**
---
# **MCP (Model Context Protocol)**

En el notebook 8.1 vimos el *function calling*: el modelo decide qué función hay que ejecutar y con qué argumentos, y nuestro programa la ejecuta. En este notebook damos un paso más con **MCP**, un estándar para ofrecer herramientas a cualquier aplicación con modelos de lenguaje.

Al terminar sabrás:

- qué problema resuelve MCP y cuáles son sus piezas (host, cliente y servidor),
- distinguir las tres primitivas de MCP: herramientas, recursos y *prompts*,
- crear un servidor MCP en Python con **FastMCP**,
- conectarte a él con un cliente y usar sus herramientas,
- hacer que un modelo de lenguaje use esas herramientas.

> Usaremos un modelo local en **LM Studio**, igual que en el notebook 10.2, y **FastMCP 4.2**. Probado en noviembre de 2026.

## **1. ¿Qué problema resuelve MCP?**

Con *function calling*, las herramientas se definen **dentro** de cada aplicación. Si queremos que ChatGPT, Claude, VS Code y nuestro propio *chatbot* consulten la misma base de datos, hay que programar la integración cuatro veces, una para cada aplicación.

**MCP** (*Model Context Protocol*) es un protocolo abierto que separa las herramientas de las aplicaciones:

- Las herramientas se programan **una sola vez** en un **servidor MCP**.
- Cualquier aplicación compatible con MCP puede conectarse a ese servidor, descubrir qué herramientas ofrece y usarlas.

Se suele comparar con un puerto USB-C: un conector estándar que permite enchufar cualquier dispositivo a cualquier ordenador.

```
 Sin MCP: cada aplicación integra cada servicio      Con MCP: un protocolo común

   App A ──── Calendario                               App A ──┐            ┌── Servidor MCP: Calendario
   App A ──── Base de datos                            App B ──┼── MCP ─────┼── Servidor MCP: Base de datos
   App B ──── Calendario                               App C ──┘            └── Servidor MCP: Correo
   App B ──── Base de datos
   ...
```

## **2. Las piezas de MCP**

En MCP intervienen tres papeles:

| Pieza | Qué es | Ejemplo |
| --- | --- | --- |
| **Host** | La aplicación que usa el usuario y que contiene el modelo de lenguaje. | Claude Desktop, VS Code, LM Studio o nuestro programa en Python. |
| **Cliente** | El componente del host que se conecta a **un** servidor MCP y habla con él. | En este notebook, `fastmcp.Client`. |
| **Servidor** | El programa que ofrece las herramientas y los datos. | El servidor que programaremos con FastMCP. |

Es importante no confundirse: **el modelo de lenguaje no habla directamente con el servidor**. El modelo solo decide qué herramienta usar; es el host, a través del cliente, quien la llama y le devuelve el resultado.

```
┌──────────────────── Host ────────────────────┐
│                                              │          ┌──────────────────┐
│   Modelo de lenguaje  ◀──▶  Cliente MCP  ◀───┼─ MCP ───▶│  Servidor MCP    │
│                                              │          │  (herramientas)  │
└──────────────────────────────────────────────┘          └──────────────────┘
```

Los mensajes entre cliente y servidor siguen el formato **JSON-RPC** y pueden viajar de dos formas (*transportes*):

- **stdio**: el host arranca el servidor como un proceso en el mismo ordenador y se comunican por la entrada y la salida estándar.
- **HTTP**: el servidor escucha en una dirección web (por ejemplo, `http://localhost:8000/mcp`) y puede estar en otra máquina. Es el que usaremos.

## **3. Las tres primitivas de MCP**

Un servidor MCP puede ofrecer tres tipos de elementos:

| Primitiva | Qué es | Quién decide usarla | Ejemplo |
| --- | --- | --- | --- |
| **Herramienta** (*tool*) | Una función que realiza una acción o un cálculo. | El **modelo** | Consultar el tiempo, enviar un correo. |
| **Recurso** (*resource*) | Datos de solo lectura, identificados por una URI. | La **aplicación** | El contenido de un fichero, una ficha de la asignatura. |
| ***Prompt*** | Una plantilla de mensaje reutilizable. | El **usuario** | "Resume este texto en una frase". |

Las herramientas son, con diferencia, las más utilizadas, y son el equivalente en MCP de las funciones del notebook 8.1.

## **4. FastMCP**

<p align="center">
  <img src="imgs/f-watercolor-waves.jpeg" width="600px">
</p>

[FastMCP](https://gofastmcp.com) es la biblioteca de Python más popular para crear servidores (y clientes) MCP. Se encarga de todos los detalles del protocolo: nosotros solo escribimos funciones normales de Python y FastMCP las convierte en herramientas.

FastMCP necesita Python 3.11 o posterior. Instalamos también `openai`, que usaremos en el apartado 7:

In [ ]:
%pip install --upgrade fastmcp openai

## **5. Crear un servidor MCP**

Un servidor con FastMCP se escribe en tres pasos:

1. Crear el servidor con `FastMCP("nombre")`.
2. Decorar funciones con `@mcp.tool`, `@mcp.resource(...)` o `@mcp.prompt`.
3. Arrancarlo con `mcp.run(...)`.

Lo más importante de cada función son su **nombre**, su ***docstring*** y los **tipos** de sus parámetros. FastMCP los usa para describir la herramienta al modelo: el *docstring* le dice *para qué sirve* y los tipos, *qué argumentos necesita*. Escríbelos con cuidado, porque el modelo decidirá si usa la herramienta a partir de esa información.

La celda siguiente guarda el servidor en el fichero `mcp_server.py`:

In [1]:
%%writefile mcp_server.py
from fastmcp import FastMCP

mcp = FastMCP("Asistente de PLN")


# Herramientas: funciones que el modelo puede decidir ejecutar
@mcp.tool
def get_weather(location: str) -> dict:
    """Devuelve la temperatura actual en una ciudad (datos simulados)."""
    return {"ciudad": location, "temperatura": 25, "unidad": "°C"}


@mcp.tool
def contar_palabras(texto: str) -> int:
    """Cuenta cuántas palabras tiene un texto."""
    return len(texto.split())


# Recurso: datos de solo lectura identificados por una URI
@mcp.resource("info://asignatura")
def info_asignatura() -> str:
    """Información básica de la asignatura."""
    return "Procesamiento del Lenguaje Natural. Temas: tokenización, embeddings, transformers, agentes y MCP."


# Prompt: una plantilla de mensaje reutilizable
@mcp.prompt
def resumir(texto: str) -> str:
    """Plantilla para pedir un resumen de un texto."""
    return f"Resume el siguiente texto en una sola frase:\n\n{texto}"


if __name__ == "__main__":
    mcp.run(transport="http", port=8000)

Overwriting mcp_server.py


Fíjate en que `get_weather` devuelve **datos** (un diccionario), no una frase ya redactada. Es una buena práctica: la herramienta aporta la información y el modelo se encarga de redactar la respuesta para el usuario.

### **5.1. Arrancar el servidor**

Un servidor se queda funcionando indefinidamente, esperando peticiones. Lo normal es arrancarlo en un **terminal**:

```bash
python mcp_server.py
```

Para no salir del notebook, lo arrancaremos en segundo plano con `subprocess`. Sus mensajes se guardan en el fichero `mcp_server.log`, que puedes abrir si algo falla. Al final del notebook lo detendremos.

In [ ]:
import subprocess
import sys
import time

registro = open("mcp_server.log", "w")
servidor = subprocess.Popen([sys.executable, "mcp_server.py"], stdout=registro, stderr=registro)
time.sleep(5)  # Damos tiempo a que arranque

if servidor.poll() is None:
    print("Servidor en marcha en http://localhost:8000/mcp")
else:
    print("El servidor no ha arrancado. Revisa mcp_server.log (¿está ocupado el puerto 8000?)")

## **6. Conectarse con un cliente MCP**

Antes de meter un modelo de lenguaje, usemos el servidor directamente con un cliente. Así veremos qué ocurre "por debajo" cuando una aplicación usa MCP.

El cliente de FastMCP es **asíncrono**: sus métodos se llaman con `await` y la conexión se abre con `async with`. En Jupyter podemos usar `await` directamente en una celda.

### **6.1. Descubrir las herramientas**

Lo primero que hace un host al conectarse es preguntar qué herramientas hay:

In [1]:
from fastmcp import Client

URL_MCP = "http://localhost:8000/mcp"

async with Client(URL_MCP) as cliente:
    herramientas = await cliente.list_tools()

for herramienta in herramientas:
    print("Nombre:     ", herramienta.name)
    print("Descripción:", herramienta.description)
    print("Parámetros: ", herramienta.input_schema)
    print()

Nombre:      get_weather
Descripción: Devuelve la temperatura actual en una ciudad (datos simulados).
Parámetros:  {'type': 'object', 'additionalProperties': False, 'properties': {'location': {'type': 'string'}}, 'required': ['location']}

Nombre:      contar_palabras
Descripción: Cuenta cuántas palabras tiene un texto.
Parámetros:  {'type': 'object', 'additionalProperties': False, 'properties': {'texto': {'type': 'string'}}, 'required': ['texto']}



Observa que FastMCP ha generado la descripción a partir del *docstring* y el esquema de parámetros (`input_schema`) a partir de los tipos de Python. Es la misma información que en el notebook 8.1 escribíamos a mano en el campo `parameters`.

### **6.2. Llamar a una herramienta**

Para ejecutar una herramienta indicamos su nombre y sus argumentos en un diccionario:

In [2]:
async with Client(URL_MCP) as cliente:
    resultado = await cliente.call_tool("get_weather", {"location": "Sevilla"})

print("Datos de Python:", resultado.data)
print("Texto enviado:  ", resultado.content[0].text)

Datos de Python: {'ciudad': 'Sevilla', 'temperatura': 25, 'unidad': '°C'}
Texto enviado:   {"ciudad":"Sevilla","temperatura":25,"unidad":"°C"}


El resultado llega de dos formas:

- `resultado.data`: el valor ya convertido a Python (aquí, un diccionario).
- `resultado.content`: el contenido tal como viaja por el protocolo, en forma de texto. Es lo que se le pasa al modelo.

### **6.3. Recursos y *prompts***

Los recursos se leen por su URI y los *prompts* se rellenan con sus argumentos:

In [3]:
async with Client(URL_MCP) as cliente:
    recurso = await cliente.read_resource("info://asignatura")
    prompt = await cliente.get_prompt("resumir", {"texto": "MCP es un protocolo abierto para conectar modelos con herramientas."})

print("Recurso:", recurso[0].text)
print()
print("Prompt: ", prompt.messages[0].content.text)

Recurso: Procesamiento del Lenguaje Natural. Temas: tokenización, embeddings, transformers, agentes y MCP.

Prompt:  Resume el siguiente texto en una sola frase:

MCP es un protocolo abierto para conectar modelos con herramientas.


Fíjate en que nadie ha decidido nada "inteligente" todavía: el cliente solo ejecuta lo que le pedimos. En el siguiente apartado será el **modelo** quien decida qué herramienta usar.

## **7. Conectar el servidor con un modelo de lenguaje**

Vamos a programar un pequeño **host**: un programa que une un modelo de lenguaje con nuestro servidor MCP. Combina lo que ya sabemos del *function calling* (notebook 8.1) con el cliente MCP del apartado anterior:

1. Pedimos al servidor MCP la lista de herramientas.
2. Se la pasamos al modelo en el formato de *function calling*.
3. Si el modelo pide usar alguna herramienta, la ejecutamos **en el servidor MCP**.
4. Devolvemos los resultados al modelo para que redacte la respuesta final.

```
 Usuario ──▶ Host ──(herramientas)──▶ Modelo
                                        │ "llama a get_weather(Sevilla)"
             Host ◀─────────────────────┘
               │ call_tool
               ▼
         Servidor MCP ──(25 °C)──▶ Host ──(resultado)──▶ Modelo ──▶ "En Sevilla hace 25 °C"
```

Usamos el modelo local de LM Studio, igual que en el notebook 10.2. Arranca su servidor antes de continuar.

In [4]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODELO = "google/gemma-4-e4b"

### **7.1. Traducir las herramientas MCP al formato del modelo**

Cada herramienta MCP se convierte en una función con el formato de la API de Responses. Solo hay que copiar el nombre, la descripción y el esquema de parámetros:

In [5]:
async with Client(URL_MCP) as cliente:
    herramientas_mcp = await cliente.list_tools()

herramientas = [
    {
        "type": "function",
        "name": h.name,
        "description": h.description,
        "parameters": h.input_schema,
    }
    for h in herramientas_mcp
]

for h in herramientas:
    print(h["name"], "→", h["description"])

get_weather → Devuelve la temperatura actual en una ciudad (datos simulados).
contar_palabras → Cuenta cuántas palabras tiene un texto.


### **7.2. El bucle del host**

La función `preguntar` sigue los cuatro pasos descritos arriba. Si el modelo no necesita ninguna herramienta, devuelve directamente su respuesta.

In [6]:
import json

async def preguntar(pregunta):
    # 1. El modelo recibe la pregunta y la lista de herramientas
    respuesta = client.responses.create(
        model=MODELO,
        input=pregunta,
        tools=herramientas,
        reasoning={"effort": "none"},
    )

    # 2. ¿Ha pedido el modelo usar alguna herramienta?
    llamadas = [item for item in respuesta.output if item.type == "function_call"]
    if not llamadas:
        return respuesta.output_text

    # 3. Ejecutamos cada llamada en el servidor MCP
    resultados = []
    async with Client(URL_MCP) as cliente:
        for llamada in llamadas:
            argumentos = json.loads(llamada.arguments)
            print(f"  [MCP] {llamada.name}({argumentos})")
            resultado = await cliente.call_tool(llamada.name, argumentos)
            resultados.append({
                "type": "function_call_output",
                "call_id": llamada.call_id,
                "output": resultado.content[0].text,
            })

    # 4. El modelo redacta la respuesta final con los resultados
    respuesta_final = client.responses.create(
        model=MODELO,
        input=resultados,
        previous_response_id=respuesta.id,
        tools=herramientas,
        reasoning={"effort": "none"},
    )
    return respuesta_final.output_text

Probemos con varias preguntas. Las líneas que empiezan por `[MCP]` muestran las herramientas que el modelo ha decidido usar:

In [7]:
preguntas = [
    "¿Qué temperatura hace en Sevilla?",
    "¿Cuántas palabras tiene la frase 'el gato come pescado fresco'?",
    "¿Qué temperatura hace en Madrid y en Bilbao?",
    "¿Qué es un embedding? Responde en una frase.",
]

for pregunta in preguntas:
    print("Pregunta:", pregunta)
    print("Respuesta:", await preguntar(pregunta))
    print()

Pregunta: ¿Qué temperatura hace en Sevilla?
  [MCP] get_weather({'location': 'Sevilla'})
Respuesta: La temperatura actual en Sevilla es de 25°C.

Pregunta: ¿Cuántas palabras tiene la frase 'el gato come pescado fresco'?
  [MCP] contar_palabras({'texto': 'el gato come pescado fresco'})
Respuesta: La frase "el gato come pescado fresco" tiene 5 palabras.

Pregunta: ¿Qué temperatura hace en Madrid y en Bilbao?
  [MCP] get_weather({'location': 'Madrid'})
  [MCP] get_weather({'location': 'Bilbao'})
Respuesta: La temperatura en Madrid es de 25°C y la temperatura en Bilbao es de 25°C.

Pregunta: ¿Qué es un embedding? Responde en una frase.
Respuesta: Un *embedding* es una representación vectorial de datos (como palabras o textos) que captura su significado semántico para que puedan ser procesados por modelos de machine learning.



Observa que:

- El modelo **elige** la herramienta adecuada para cada pregunta y extrae los argumentos del texto.
- Puede pedir **varias llamadas** a la vez (Madrid y Bilbao).
- Cuando no necesita ninguna herramienta (la pregunta sobre *embeddings*), responde directamente.

Si añades una herramienta nueva a `mcp_server.py` y reinicias el servidor, el host la usará sin cambiar ni una línea de su código: esa es la gran ventaja de MCP.

## **8. MCP en otras aplicaciones**

Nuestro host es muy sencillo, pero las aplicaciones reales hacen exactamente lo mismo. Hay dos formas habituales de conectarles un servidor MCP.

### **8.1. Configurar el servidor en la aplicación**

Aplicaciones como Claude Desktop, VS Code, Cursor o el chat de LM Studio permiten añadir servidores MCP en un fichero de configuración. Su formato varía ligeramente entre aplicaciones, pero suele parecerse a este:

```json
{
  "mcpServers": {
    "asistente-pln": {
      "url": "http://localhost:8000/mcp"
    }
  }
}
```

A partir de ese momento, el modelo de la aplicación puede usar nuestras herramientas en el chat.

### **8.2. Dejar que la API llame al servidor**

La API de OpenAI (y también LM Studio) permite pasar un servidor MCP directamente como herramienta. En ese caso es el **propio servicio** quien hace de host: se conecta al servidor, llama a las herramientas y devuelve la respuesta final.

```python
respuesta = client.responses.create(
    model="...",
    tools=[{
        "type": "mcp",
        "server_label": "asistente_pln",
        "server_url": "https://mi-servidor.ejemplo.com/mcp",
        "require_approval": "never",
    }],
    input="¿Qué temperatura hace en Sevilla?",
)
```

La limitación es que el servidor MCP tiene que tener una **dirección pública** en internet: el servicio no puede acceder a `localhost`. LM Studio, por ejemplo, rechaza las direcciones locales con el error *"URL resolves to a non-public address"*. Para pruebas puede usarse una herramienta como [ngrok](https://ngrok.com), que crea una dirección pública temporal que redirige a nuestro ordenador:

```bash
ngrok http 8000
```

> **Cuidado**: con una dirección pública, cualquiera que la conozca puede usar tus herramientas. No expongas así herramientas que accedan a datos privados o que realicen acciones peligrosas, y ten en cuenta que `"require_approval": "never"` deja que el modelo las ejecute sin pedir confirmación.

## **9. Detener el servidor**

Cuando termines, detén el servidor que arrancamos en el apartado 5.1:

In [ ]:
servidor.terminate()
servidor.wait()
registro.close()
print("Servidor detenido")

## **10. Resumen**

```python
# Servidor (mcp_server.py)
from fastmcp import FastMCP
mcp = FastMCP("Mi servidor")

@mcp.tool
def mi_herramienta(parametro: str) -> dict:
    """Descripción que leerá el modelo."""
    return {...}

mcp.run(transport="http", port=8000)

# Cliente
from fastmcp import Client
async with Client("http://localhost:8000/mcp") as cliente:
    herramientas = await cliente.list_tools()
    resultado = await cliente.call_tool("mi_herramienta", {"parametro": "..."})

# Host: herramientas MCP → function calling
tools = [{"type": "function", "name": h.name, "description": h.description,
          "parameters": h.input_schema} for h in herramientas]
```

## **11. Ejercicios**

1. **Nueva herramienta.** Añade al servidor una herramienta `traducir_numero(n: int) -> str` que devuelva un número escrito en letras (del 0 al 10). Reinicia el servidor y comprueba que el host la usa sin modificar su código.
2. **Docstrings.** Cambia el *docstring* de `contar_palabras` por algo poco claro (por ejemplo, `"Función auxiliar."`). ¿Sigue eligiéndola el modelo? ¿Qué conclusión sacas?
3. **Errores.** Haz que `get_weather` lance un `ValueError` si la ciudad es una cadena vacía. ¿Qué devuelve `call_tool`? ¿Cómo debería tratarlo el host?
4. **Recursos en el host.** Modifica `preguntar` para que lea el recurso `info://asignatura` y lo añada como instrucciones del modelo. Pregunta después: *"¿Qué temas tiene la asignatura?"*.
5. **Datos reales.** Sustituye los datos simulados de `get_weather` por una consulta a una API meteorológica gratuita, como [Open-Meteo](https://open-meteo.com).

## **Referencias**

- [Documentación de MCP](https://modelcontextprotocol.io)
- [Documentación de FastMCP](https://gofastmcp.com)
- [Repositorio de FastMCP](https://github.com/PrefectHQ/fastmcp)
- [Herramientas MCP en la API de OpenAI](https://developers.openai.com/api/docs/guides/tools-connectors-mcp)
- [LM Studio: API compatible con OpenAI](https://lmstudio.ai/docs/developer/openai-compat)